# Exploratory Data Analysis — JogjaPulse (Mobilitas Device di DIY, Des 2021 – Apr 2022)

**Tujuan:** memahami struktur, kualitas, dan pola awal dari seluruh file data JogjaPulse sebelum analisis lanjutan.
Tahap ini sengaja **tanpa visualisasi**. Semua temuan disajikan dalam bentuk tabel dan statistik.

**Dataset yang dianalisis**

| Bagian | File | Isi |
|---|---|---|
| A | `presence_kapanewon_hourly.csv` | Device per kapanewon per jam |
| B | `presence_cell500m_daily.csv` | Device per grid 500 m per hari |
| C | `od_home_visits_kapanewon_monthly.csv` + `od_origin_homes_monthly.csv` | Aliran asal–tujuan bulanan + penyebutnya |
| D | `home_maids_vs_population_kapanewon.csv` | Device ber-home vs jumlah penduduk |
| E | `validation_night_presence_kapanewon.csv` | Presence malam hari (validasi) |
| F | `tier1_summary.csv`, `od_summary.csv` | Ringkasan suppression (cek konsistensi) |

**Hal penting dari README yang memengaruhi EDA**
- `n_maids` = jumlah device **distinct** → **tidak boleh dijumlahkan** antar jam/wilayah/hari.
- Baris yang tidak ada = jumlah < 5 (suppressed, k = 5), **bukan nol**.
- Jumlah device di sumber data turun drastis dari Desember ke April → perbandingan antar waktu perlu dinormalisasi.
- Data OD Maret–April 2022 berkualitas rendah (`reduced_quality = True`).

## 0. Setup dan memuat data

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:,.3f}".format)

# Ganti jika file dipindah
DATA_DIR = Path(".")

In [ ]:
hourly = pd.read_csv(DATA_DIR / "presence_kapanewon_hourly.csv", parse_dates=["date"],
                     dtype={"kab_code": str, "kapanewon_code": str})
cell = pd.read_csv(DATA_DIR / "presence_cell500m_daily.csv", parse_dates=["date"])
od = pd.read_csv(DATA_DIR / "od_home_visits_kapanewon_monthly.csv",
                 dtype={"origin_kab_code": str, "origin_code": str,
                        "dest_kab_code": str, "dest_code": str})
origin = pd.read_csv(DATA_DIR / "od_origin_homes_monthly.csv",
                     dtype={"origin_kab_code": str, "origin_code": str})
homepop = pd.read_csv(DATA_DIR / "home_maids_vs_population_kapanewon.csv")
night = pd.read_csv(DATA_DIR / "validation_night_presence_kapanewon.csv")
tier1 = pd.read_csv(DATA_DIR / "tier1_summary.csv")
od_sum = pd.read_csv(DATA_DIR / "od_summary.csv")

datasets = {
    "presence_kapanewon_hourly": hourly,
    "presence_cell500m_daily": cell,
    "od_home_visits_kapanewon_monthly": od,
    "od_origin_homes_monthly": origin,
    "home_maids_vs_population_kapanewon": homepop,
    "validation_night_presence_kapanewon": night,
    "tier1_summary": tier1,
    "od_summary": od_sum,
}

### 0.1 Gambaran umum semua file

In [ ]:
overview = pd.DataFrame({
    name: {
        "baris": d.shape[0],
        "kolom": d.shape[1],
        "memori_MB": d.memory_usage(deep=True).sum() / 1e6,
        "total_sel_null": int(d.isna().sum().sum()),
        "baris_duplikat": int(d.duplicated().sum()),
    }
    for name, d in datasets.items()
}).T
overview

,baris,kolom,memori_MB,total_sel_null,baris_duplikat
presence_kapanewon_hourly,"230,418.000",8.000,59.183,0.000,0.000
presence_cell500m_daily,"183,548.000",7.000,19.823,0.000,0.000
od_home_visits_kapanewon_monthly,"9,894.000",11.000,5.085,0.000,0.000
od_origin_homes_monthly,390.000,8.000,0.116,0.000,0.000
home_maids_vs_population_kapanewon,78.000,5.000,0.011,0.000,0.000
validation_night_presence_kapanewon,156.000,7.000,0.034,0.000,0.000
tier1_summary,2.000,8.000,0.000,0.000,0.000
od_summary,1.000,7.000,0.000,0.000,0.000


In [ ]:
def profil_kolom(df):
    """Ringkasan per kolom: tipe data, null, jumlah nilai unik, contoh nilai."""
    return pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "n_null": df.isna().sum(),
        "pct_null": df.isna().mean() * 100,
        "n_unik": df.nunique(),
        "contoh": df.iloc[0],
    })

---
## A. `presence_kapanewon_hourly` — device per kapanewon per jam

### A.1 Struktur dan integritas

In [ ]:
display(hourly.head())
profil_kolom(hourly)

,kab_code,kab_kota,kapanewon_code,kapanewon,date,hour,n_maids,n_pings
0,34.01,KULON PROGO,34.01.01,TEMON,2021-12-01,0,16,66
1,34.01,KULON PROGO,34.01.02,WATES,2021-12-01,0,58,101
2,34.01,KULON PROGO,34.01.03,PANJATAN,2021-12-01,0,11,32
3,34.01,KULON PROGO,34.01.04,GALUR,2021-12-01,0,9,50
4,34.01,KULON PROGO,34.01.05,LENDAH,2021-12-01,0,14,74


,dtype,n_null,pct_null,n_unik,contoh
kab_code,object,0,0.000,5,34.01
kab_kota,object,0,0.000,5,KULON PROGO
kapanewon_code,object,0,0.000,78,34.01.01
kapanewon,object,0,0.000,77,TEMON
date,datetime64[ns],0,0.000,151,2021-12-01 00:00:00
hour,int64,0,0.000,24,0
n_maids,int64,0,0.000,1298,16
n_pings,int64,0,0.000,3195,66


In [ ]:
n_days = hourly["date"].nunique()
n_kap = hourly["kapanewon_code"].nunique()
key = ["kapanewon_code", "date", "hour"]

print("Rentang tanggal     :", hourly["date"].min().date(), "s.d.", hourly["date"].max().date())
print("Jumlah hari unik    :", n_days, "(seharusnya 151)")
print("Jumlah kapanewon    :", n_kap, "(seharusnya 78)")
print("Jam unik            :", sorted(hourly["hour"].unique()))
print("Duplikat kunci      :", hourly.duplicated(key).sum())
print("n_maids minimum     :", hourly["n_maids"].min(), "(seharusnya >= 5)")
print("Baris n_pings < n_maids:", (hourly["n_pings"] < hourly["n_maids"]).sum(),
      "(seharusnya 0: tiap device minimal 1 ping)")
print("Kode kapanewon dengan >1 nama:",
      (hourly.groupby("kapanewon_code")["kapanewon"].nunique() > 1).sum())
print("Nama kapanewon yang dipakai >1 kode:",
      hourly.groupby("kapanewon")["kapanewon_code"].nunique().loc[lambda s: s > 1].to_dict())

Rentang tanggal     : 2021-12-01 s.d. 2022-04-30
Jumlah hari unik    : 151 (seharusnya 151)
Jumlah kapanewon    : 78 (seharusnya 78)
Jam unik            : [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20), np.int64(21), np.int64(22), np.int64(23)]
Duplikat kunci      : 0
n_maids minimum     : 5 (seharusnya >= 5)
Baris n_pings < n_maids: 0 (seharusnya 0: tiap device minimal 1 ping)
Kode kapanewon dengan >1 nama: 0
Nama kapanewon yang dipakai >1 kode: {'JETIS': 2}


> Catatan: nama kapanewon tidak unik (contoh: **JETIS** ada di Bantul dan Kota Yogyakarta).
> Untuk join dan groupby, gunakan **kode** atau pasangan (`kab_kota`, `kapanewon`).

### A.2 Kelengkapan data (efek suppression k = 5)

Idealnya ada 78 kapanewon × 151 hari × 24 jam slot. Slot yang tidak ada berarti device < 5.

In [ ]:
hourly["month"] = hourly["date"].dt.to_period("M")

expected_total = n_kap * n_days * 24
print(f"Slot seharusnya : {expected_total:,}")
print(f"Slot terisi     : {len(hourly):,}")
print(f"Slot hilang     : {1 - len(hourly) / expected_total:.1%}")

Slot seharusnya : 282,672
Slot terisi     : 230,418
Slot hilang     : 18.5%


In [ ]:
kap_per_kab = hourly.groupby("kab_kota")["kapanewon_code"].nunique()
days_per_month = hourly.groupby("month")["date"].nunique()

# % slot terisi per kabupaten x bulan
slots = hourly.groupby(["kab_kota", "month"]).size().unstack()
expected = pd.DataFrame(np.outer(kap_per_kab, days_per_month * 24),
                        index=kap_per_kab.index, columns=days_per_month.index)
pct_terisi_kab_bulan = slots / expected * 100
pct_terisi_kab_bulan["SEMUA"] = slots.sum(axis=1) / expected.sum(axis=1) * 100
pct_terisi_kab_bulan

month,2021-12,2022-01,2022-02,2022-03,2022-04,SEMUA
kab_kota,,,,,,
BANTUL,99.217,98.458,95.439,77.253,65.850,87.222
GUNUNGKIDUL,84.013,76.314,71.768,41.159,23.619,59.365
KULON PROGO,98.017,96.539,92.584,68.627,41.748,79.493
SLEMAN,99.960,99.945,98.766,89.485,83.693,94.353
YOGYAKARTA,92.867,95.046,97.927,82.728,77.927,89.203


In [ ]:
# % slot terisi per jam (semua kapanewon, semua hari)
pct_terisi_jam = (hourly.groupby("hour").size() / (n_kap * n_days) * 100).rename("pct_slot_terisi")
pct_terisi_jam.to_frame().T

hour,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23
pct_slot_terisi,70.368,63.670,59.526,60.197,66.293,75.004,80.701,83.945,86.076,87.129,87.247,88.793,89.209,88.529,88.088,87.511,87.536,88.266,90.211,89.056,89.812,87.511,83.970,77.696


In [ ]:
# 10 kapanewon dengan kelengkapan terendah
lengkap_kap = (hourly.groupby(["kab_kota", "kapanewon"]).size() / (n_days * 24) * 100).rename("pct_slot_terisi")
print("Kapanewon dengan slot terisi < 50%:", (lengkap_kap < 50).sum())
lengkap_kap.sort_values().head(10).to_frame()

Kapanewon dengan slot terisi < 50%: 7


pct_slot_terisi
kab_kota    kapanewon                   
GUNUNGKIDUL GIRISUBO              21.634
            RONGKOP               27.594
            PURWOSARI             35.430
            TEPUS                 42.191
            TANJUNGSARI           43.902
            SAPTOSARI             45.419
            PANGGANG              49.834
            NGAWEN                53.091
KULON PROGO PANJATAN              63.438
GUNUNGKIDUL PALIYAN               63.659

### A.3 Statistik deskriptif

In [ ]:
hourly["ping_per_maid"] = hourly["n_pings"] / hourly["n_maids"]
num_cols = ["n_maids", "n_pings", "ping_per_maid"]

desc = hourly[num_cols].describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]).T
desc["skew"] = hourly[num_cols].skew()
desc["kurtosis"] = hourly[num_cols].kurt()
desc

,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max,skew,kurtosis
n_maids,"230,418.000",72.006,124.179,5.000,5.000,6.000,12.000,29.000,77.000,279.000,637.000,"2,506.000",4.782,36.049
n_pings,"230,418.000",191.423,369.298,5.000,8.000,11.000,28.000,72.000,201.000,736.000,"1,828.000","10,210.000",6.185,64.930
ping_per_maid,"230,418.000",2.605,1.151,1.000,1.111,1.333,1.875,2.390,3.046,4.516,6.667,39.375,3.400,35.537


In [ ]:
# Sebaran n_maids dalam bentuk tabel frekuensi (pengganti histogram)
bins = [5, 10, 20, 50, 100, 250, 500, 1000, np.inf]
freq = pd.cut(hourly["n_maids"], bins=bins, right=False).value_counts().sort_index().to_frame("jumlah_baris")
freq["pct"] = freq["jumlah_baris"] / freq["jumlah_baris"].sum() * 100
freq["pct_kumulatif"] = freq["pct"].cumsum()
freq

,jumlah_baris,pct,pct_kumulatif
n_maids,,,
"[5.0, 10.0)",42436,18.417,18.417
"[10.0, 20.0)",47078,20.432,38.849
"[20.0, 50.0)",57556,24.979,63.827
"[50.0, 100.0)",39261,17.039,80.867
"[100.0, 250.0)",30440,13.211,94.077
"[250.0, 500.0)",9278,4.027,98.104
"[500.0, 1000.0)",3853,1.672,99.776
"[1000.0, inf)",516,0.224,100.000


### A.4 Tren waktu dan anomali

`n_pings` bersifat aditif, jadi **total ping per hari** bisa dipakai sebagai indikator volume data harian.
`n_maids` tidak aditif, sehingga yang ditampilkan adalah rata-rata per baris.

In [ ]:
daily = hourly.groupby("date").agg(
    baris=("n_maids", "size"),
    mean_maids=("n_maids", "mean"),
    max_maids=("n_maids", "max"),
    total_pings=("n_pings", "sum"),
)

monthly = hourly.groupby("month").agg(
    baris=("n_maids", "size"),
    mean_maids=("n_maids", "mean"),
    median_maids=("n_maids", "median"),
    total_pings=("n_pings", "sum"),
    mean_ping_per_maid=("ping_per_maid", "mean"),
)
monthly["pings_per_hari"] = monthly["total_pings"] / days_per_month
monthly["indeks_mean_maids_vs_des"] = monthly["mean_maids"] / monthly["mean_maids"].iloc[0] * 100
monthly["indeks_pings_per_hari_vs_des"] = monthly["pings_per_hari"] / monthly["pings_per_hari"].iloc[0] * 100
monthly

,baris,mean_maids,median_maids,total_pings,mean_ping_per_maid,pings_per_hari,indeks_mean_maids_vs_des,indeks_pings_per_hari_vs_des
month,,,,,,,,
2021-12,54867,117.467,60.000,17977686,2.859,"579,925.355",100.000,100.000
2022-01,53833,82.203,44.000,10604631,2.382,"342,084.871",69.979,58.988
2022-02,47546,85.208,38.000,10969230,2.678,"391,758.214",72.538,67.553
2022-03,41345,23.517,14.000,2781511,2.551,"89,726.161",20.020,15.472
2022-04,32827,21.251,12.000,1774215,2.511,"59,140.500",18.091,10.198


In [ ]:
# Hari dengan penurunan total ping paling tajam dibanding hari sebelumnya
daily["pct_change_pings"] = daily["total_pings"].pct_change() * 100
daily.sort_values("pct_change_pings").head(8)

,baris,mean_maids,max_maids,total_pings,pct_change_pings
date,,,,,
2022-02-16,1653,50.139,1003,193179,-65.758
2022-03-08,993,15.817,130,37757,-59.965
2022-02-20,1656,36.528,372,162397,-49.573
2022-03-01,1434,26.990,789,97869,-38.735
2022-04-03,777,20.041,303,37055,-37.900
2022-04-19,758,22.763,144,41955,-36.889
2022-01-01,1736,73.124,818,325861,-32.788
2022-01-07,1692,74.064,569,312937,-29.111


In [ ]:
# Tanggal-tanggal yang disebut di README
cek = pd.to_datetime(["2021-12-31", "2022-01-01", "2022-02-15", "2022-02-16",
                      "2022-03-07", "2022-03-08", "2022-03-09"])
daily.loc[daily.index.intersection(cek)]

,baris,mean_maids,max_maids,total_pings,pct_change_pings
2021-12-31,1773,120.710,2094,484827,-19.008
2022-01-01,1736,73.124,818,325861,-32.788
2022-02-15,1778,115.635,1199,564162,22.745
2022-02-16,1653,50.139,1003,193179,-65.758
2022-03-07,1304,23.143,1054,94311,-13.548
2022-03-08,993,15.817,130,37757,-59.965
2022-03-09,1307,23.630,756,86048,127.899


In [ ]:
# Deteksi hari anomali: total ping < 70% dari median 7 hari di sekitarnya
daily["rasio_vs_median7"] = daily["total_pings"] / daily["total_pings"].rolling(7, center=True, min_periods=3).median()
daily[daily["rasio_vs_median7"] < 0.7][["total_pings", "rasio_vs_median7"]]

,total_pings,rasio_vs_median7
date,,
2022-02-16,193179,0.534
2022-03-08,37757,0.400
2022-04-19,41955,0.635


### A.5 Pola diurnal dan mingguan (setelah normalisasi)

Supaya tren penurunan device tidak mengacaukan pola, nilai tiap jam dibagi dengan rata-rata harian pada hari yang sama.
Indeks 1,0 = rata-rata hari itu.

In [ ]:
# Baris = tanggal, kolom = jam. Nilai = jumlah n_maids semua kapanewon (indikator kasar, bukan jumlah orang)
dh = hourly.groupby(["date", "hour"])["n_maids"].sum().unstack()
profil_jam = dh.div(dh.mean(axis=1), axis=0).agg(["mean", "std"]).T
profil_jam.columns = ["indeks_relatif", "std"]
profil_jam.T

hour,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23
indeks_relatif,0.593,0.658,0.543,0.477,0.532,0.650,0.800,0.889,1.048,1.094,1.136,1.255,1.300,1.276,1.250,1.219,1.221,1.261,1.330,1.287,1.278,1.155,0.995,0.752
std,0.162,0.432,0.252,0.253,0.282,0.167,0.192,0.135,0.161,0.166,0.194,0.126,0.146,0.156,0.150,0.146,0.141,0.141,0.137,0.176,0.185,0.169,0.138,0.136


In [ ]:
# Pola hari dalam seminggu: total ping dibagi rata-rata bergulir 7 hari (menghilangkan tren)
weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
daily["hari"] = daily.index.day_name()
daily["indeks_vs_rolling7"] = daily["total_pings"] / daily["total_pings"].rolling(7, center=True, min_periods=7).mean()

pd.DataFrame({
    "indeks_ternormalisasi": daily.groupby("hari")["indeks_vs_rolling7"].mean(),
    "std": daily.groupby("hari")["indeks_vs_rolling7"].std(),
    "mean_maids_mentah": hourly.groupby(hourly["date"].dt.day_name())["n_maids"].mean(),
    "jumlah_hari": daily.groupby("hari").size(),
}).reindex(weekday_order)

,indeks_ternormalisasi,std,mean_maids_mentah,jumlah_hari
Monday,1.006,0.102,72.907,21
Tuesday,0.961,0.200,74.130,21
Wednesday,0.986,0.154,70.428,22
Thursday,1.038,0.122,73.815,22
Friday,1.036,0.114,73.332,22
Saturday,1.014,0.124,69.484,22
Sunday,0.939,0.140,70.046,21


### A.6 Pola spasial

Agar perbandingan antar kapanewon adil, `n_maids` dinormalisasi menjadi **share**: proporsi terhadap jumlah `n_maids`
semua kapanewon pada tanggal dan jam yang sama. Analisis spasial dibatasi ke **Des 2021 – Feb 2022** karena kelengkapannya jauh lebih baik.

In [ ]:
hourly["share"] = hourly["n_maids"] / hourly.groupby(["date", "hour"])["n_maids"].transform("sum")
hq = hourly[hourly["date"] < "2022-03-01"]
n_days_hq = hq["date"].nunique()

spasial = hq.groupby(["kab_kota", "kapanewon"]).agg(
    mean_maids=("n_maids", "mean"),
    median_maids=("n_maids", "median"),
    max_maids=("n_maids", "max"),
    mean_share_pct=("share", lambda s: s.mean() * 100),
    pct_slot_terisi=("n_maids", lambda s: len(s) / (n_days_hq * 24) * 100),
)
print("10 kapanewon dengan presence tertinggi")
display(spasial.sort_values("mean_share_pct", ascending=False).head(10))
print("10 kapanewon dengan presence terendah")
display(spasial.sort_values("mean_share_pct").head(10))

10 kapanewon dengan presence tertinggi


mean_maids  median_maids  max_maids  mean_share_pct  pct_slot_terisi
kab_kota   kapanewon                                                                        
SLEMAN     DEPOK           551.781       537.500       1532           8.392          100.000
           NGAGLIK         409.676       366.000       2506           5.973          100.000
BANTUL     BANGUNTAPAN     391.595       376.000       1100           5.624          100.000
SLEMAN     GAMPING         398.096       327.500       1583           5.484          100.000
BANTUL     KASIHAN         328.500       324.500        996           4.633          100.000
SLEMAN     MLATI           237.876       225.500        815           3.481          100.000
YOGYAKARTA UMBULHARJO      223.588       219.000        839           3.321          100.000
SLEMAN     NGEMPLAK        217.862       220.000        548           3.096           99.954
           PAKEM           225.205       121.000       1495           2.989           99.954
           SEYEGAN         226.456       113.000       1089           2.987           99.537

10 kapanewon dengan presence terendah


mean_maids  median_maids  max_maids  mean_share_pct  pct_slot_terisi
kab_kota    kapanewon                                                                        
GUNUNGKIDUL GIRISUBO          7.077         6.000         19           0.087           33.009
            RONGKOP           8.376         8.000         21           0.098           45.926
            PURWOSARI         9.289         9.000         29           0.112           57.639
            TEPUS             9.522         9.000         23           0.124           63.009
            TANJUNGSARI      10.094         9.000         34           0.125           64.583
            SAPTOSARI        10.180        10.000         28           0.136           66.389
            PANGGANG         11.095        10.000         35           0.143           70.926
            NGAWEN           12.254        11.000         38           0.163           75.370
            SEMIN            14.344        13.000         45           0.207           84.491
            PATUK            15.582        14.000         47           0.223           86.250

In [ ]:
# Ringkasan per kabupaten/kota
spasial.groupby("kab_kota").agg(
    jumlah_kapanewon=("mean_maids", "size"),
    mean_maids=("mean_maids", "mean"),
    total_share_pct=("mean_share_pct", "sum"),
    rata2_slot_terisi=("pct_slot_terisi", "mean"),
).sort_values("total_share_pct", ascending=False)

,jumlah_kapanewon,mean_maids,total_share_pct,rata2_slot_terisi
kab_kota,,,,
SLEMAN,17,191.810,47.010,99.583
BANTUL,17,106.448,25.658,97.781
YOGYAKARTA,14,67.481,14.765,95.192
KULON PROGO,12,45.478,7.805,95.818
GUNUNGKIDUL,18,24.582,6.080,77.551


In [ ]:
# Rasio siang (09-16) vs malam (00-05): >1 = lebih ramai siang (pusat kegiatan), <1 = cenderung permukiman
hq = hq.assign(periode=np.select([hq["hour"].between(0, 5), hq["hour"].between(9, 16)],
                                 ["malam", "siang"], default="lain"))
dn = (hq[hq["periode"] != "lain"]
      .groupby(["kab_kota", "kapanewon", "periode"])["share"].mean().unstack())
dn["rasio_siang_malam"] = dn["siang"] / dn["malam"]
print("Rasio siang/malam tertinggi (indikasi pusat kegiatan)")
display(dn.sort_values("rasio_siang_malam", ascending=False).head(10))
print("Rasio siang/malam terendah (indikasi permukiman)")
display(dn.sort_values("rasio_siang_malam").head(10))

Rasio siang/malam tertinggi (indikasi pusat kegiatan)


periode                malam  siang  rasio_siang_malam
kab_kota    kapanewon                                 
GUNUNGKIDUL PLAYEN     0.006  0.008              1.416
            NGLIPAR    0.003  0.004              1.341
KULON PROGO SAMIGALUH  0.005  0.007              1.336
GUNUNGKIDUL WONOSARI   0.015  0.020              1.301
            PONJONG    0.003  0.004              1.291
KULON PROGO NANGGULAN  0.006  0.008              1.270
BANTUL      PANDAK     0.011  0.014              1.251
KULON PROGO KOKAP      0.008  0.009              1.226
SLEMAN      PRAMBANAN  0.009  0.011              1.211
KULON PROGO SENTOLO    0.008  0.009              1.210

Rasio siang/malam terendah (indikasi permukiman)


periode                  malam  siang  rasio_siang_malam
kab_kota    kapanewon                                   
YOGYAKARTA  GONDOMANAN   0.039  0.014              0.371
GUNUNGKIDUL GIRISUBO     0.001  0.001              0.609
SLEMAN      SLEMAN       0.033  0.024              0.718
GUNUNGKIDUL RONGKOP      0.001  0.001              0.743
            PURWOSARI    0.001  0.001              0.755
YOGYAKARTA  KRATON       0.008  0.006              0.845
            MANTRIJERON  0.006  0.005              0.846
SLEMAN      DEPOK        0.093  0.081              0.870
            NGEMPLAK     0.033  0.030              0.890
YOGYAKARTA  WIROBRAJAN   0.004  0.004              0.900

### A.7 Korelasi antar variabel

In [ ]:
pd.concat({
    "pearson": hourly[num_cols].corr(method="pearson"),
    "spearman": hourly[num_cols].corr(method="spearman"),
})

n_maids  n_pings  ping_per_maid
pearson  n_maids          1.000    0.915          0.027
         n_pings          0.915    1.000          0.191
         ping_per_maid    0.027    0.191          1.000
spearman n_maids          1.000    0.956          0.130
         n_pings          0.956    1.000          0.394
         ping_per_maid    0.130    0.394          1.000

---
## B. `presence_cell500m_daily` — device per grid 500 m per hari

### B.1 Struktur dan integritas

In [ ]:
display(cell.head())
profil_kolom(cell)

,cell_id,x_min_utm,y_min_utm,centroid_lon,centroid_lat,date,n_maids
0,E786_N18255,393000,9127500,110.032,-7.890,2021-12-01,9
1,E789_N18259,394500,9129500,110.045,-7.872,2021-12-01,7
2,E790_N18255,395000,9127500,110.050,-7.890,2021-12-01,17
3,E791_N18254,395500,9127000,110.054,-7.894,2021-12-01,5
4,E792_N18252,396000,9126000,110.059,-7.904,2021-12-01,20


,dtype,n_null,pct_null,n_unik,contoh
cell_id,object,0,0.000,5715,E786_N18255
x_min_utm,int64,0,0.000,167,393000
y_min_utm,int64,0,0.000,129,9127500
centroid_lon,float64,0,0.000,5715,110.032
centroid_lat,float64,0,0.000,5715,-7.890
date,datetime64[ns],0,0.000,151,2021-12-01 00:00:00
n_maids,int64,0,0.000,1319,9


In [ ]:
n_cells = cell["cell_id"].nunique()
print("Rentang tanggal     :", cell["date"].min().date(), "s.d.", cell["date"].max().date())
print("Jumlah hari unik    :", cell["date"].nunique())
print("Jumlah sel unik     :", n_cells, "(README: 5.715)")
print("Duplikat cell_id+date:", cell.duplicated(["cell_id", "date"]).sum())
print("n_maids minimum     :", cell["n_maids"].min())

# Konsistensi cell_id dengan koordinat UTM: E{i}_N{j} -> x = i*500, y = j*500
ij = cell["cell_id"].str.extract(r"E(\d+)_N(\d+)").astype(int)
print("cell_id konsisten dgn x_min_utm:", (ij[0] * 500 == cell["x_min_utm"]).all())
print("cell_id konsisten dgn y_min_utm:", (ij[1] * 500 == cell["y_min_utm"]).all())
print("Satu sel = satu koordinat:", (cell.groupby("cell_id")[["centroid_lon", "centroid_lat"]].nunique() == 1).all().all())

cell[["centroid_lon", "centroid_lat"]].describe().T

Rentang tanggal     : 2021-12-01 s.d. 2022-04-30
Jumlah hari unik    : 151
Jumlah sel unik     : 5715 (README: 5.715)
Duplikat cell_id+date: 0
n_maids minimum     : 5
cell_id konsisten dgn x_min_utm: True
cell_id konsisten dgn y_min_utm: True
Satu sel = satu koordinat: True


,count,mean,std,min,25%,50%,75%,max
centroid_lon,"183,548.000",110.370,0.114,110.032,110.322,110.372,110.417,110.816
centroid_lat,"183,548.000",-7.811,0.089,-8.176,-7.863,-7.800,-7.750,-7.579


### B.2 Kelengkapan dan aktivitas sel

README menyebut 64,6% baris sel × hari di-suppress. Di sini dilihat berapa sel yang muncul per hari dan berapa hari tiap sel muncul.

In [ ]:
cell["month"] = cell["date"].dt.to_period("M")
print(f"Kombinasi sel x hari jika semua sel muncul tiap hari: {n_cells * cell['date'].nunique():,}")
print(f"Baris yang ada: {len(cell):,} ({len(cell) / (n_cells * cell['date'].nunique()):.1%})")

daily_cells = cell.groupby("date").agg(
    n_sel=("cell_id", "size"),
    mean_maids=("n_maids", "mean"),
    median_maids=("n_maids", "median"),
    max_maids=("n_maids", "max"),
)
daily_cells.groupby(daily_cells.index.to_period("M")).agg(["mean", "min", "max"])

Kombinasi sel x hari jika semua sel muncul tiap hari: 862,965
Baris yang ada: 183,548 (21.3%)


n_sel             mean_maids               median_maids               max_maids             
             mean   min   max       mean    min    max         mean    min    max      mean   min    max
date                                                                                                    
2021-12 1,563.871  1373  1778     67.168 29.585 87.386       16.548 13.000 18.000 5,432.484  2853  10971
2022-01 1,771.387  1334  2616     47.809 36.614 61.006       15.597 14.000 18.000 1,640.387   819   3257
2022-02 1,888.536   690  2729     35.902 18.824 53.583       12.929  9.000 16.000 2,662.071   837   7899
2022-03   519.839   361   699     22.184 15.673 29.142        8.129  7.000  9.000 2,076.323   844   3760
2022-04   372.033   266   461     23.483 19.634 30.326        8.000  7.000  9.000 1,210.333   903   1885

In [ ]:
hari_aktif = cell.groupby("cell_id")["date"].nunique()
display(hari_aktif.describe().to_frame("hari_aktif_per_sel").T)

freq_aktif = pd.cut(hari_aktif, bins=[0, 1, 7, 30, 90, 150, 151],
                    labels=["1 hari", "2-7", "8-30", "31-90", "91-150", "151 (selalu)"]).value_counts().sort_index()
freq_aktif.to_frame("jumlah_sel").assign(pct=lambda d: d["jumlah_sel"] / d["jumlah_sel"].sum() * 100)

,count,mean,std,min,25%,50%,75%,max
hari_aktif_per_sel,"5,715.000",32.117,38.896,1.000,3.000,14.000,49.000,151.000


,jumlah_sel,pct
date,,
1 hari,911,15.941
2-7,1366,23.902
8-30,1351,23.640
31-90,1538,26.912
91-150,482,8.434
151 (selalu),67,1.172


### B.3 Statistik deskriptif `n_maids` per sel

In [ ]:
d = cell["n_maids"].describe(percentiles=[.25, .5, .75, .9, .95, .99]).to_frame().T
d["skew"] = cell["n_maids"].skew()
d["kurtosis"] = cell["n_maids"].kurt()
display(d)

freq = pd.cut(cell["n_maids"], bins=[5, 10, 20, 50, 100, 250, 500, np.inf], right=False).value_counts().sort_index().to_frame("jumlah_baris")
freq["pct"] = freq["jumlah_baris"] / freq["jumlah_baris"].sum() * 100
freq

,count,mean,std,min,25%,50%,75%,90%,95%,99%,max,skew,kurtosis
n_maids,"183,548.000",46.667,151.808,5.000,7.000,13.000,34.000,96.000,182.000,565.000,"10,971.000",21.565,874.929


,jumlah_baris,pct
n_maids,,
"[5.0, 10.0)",68637,37.395
"[10.0, 20.0)",44946,24.487
"[20.0, 50.0)",36629,19.956
"[50.0, 100.0)",15677,8.541
"[100.0, 250.0)",11371,6.195
"[250.0, 500.0)",3963,2.159
"[500.0, inf)",2325,1.267


In [ ]:
# Konsentrasi: berapa persen "device-hari" yang ada di 1% / 10% sel teratas
# (jumlah n_maids per sel di sini dipakai hanya sebagai indikator konsentrasi relatif)
per_sel = cell.groupby("cell_id")["n_maids"].sum().sort_values(ascending=False)
for p in [0.01, 0.05, 0.10, 0.25]:
    k = max(1, int(len(per_sel) * p))
    print(f"Top {p:>4.0%} sel ({k:>4} sel) menampung {per_sel.iloc[:k].sum() / per_sel.sum():.1%} device-hari")

Top   1% sel (  57 sel) menampung 33.6% device-hari
Top   5% sel ( 285 sel) menampung 66.2% device-hari
Top  10% sel ( 571 sel) menampung 79.9% device-hari
Top  25% sel (1428 sel) menampung 93.9% device-hari


In [ ]:
# 10 sel paling ramai (Des-Feb) beserta koordinatnya -> bisa dicek di peta (Malioboro, kampus, dll.)
cq = cell[cell["date"] < "2022-03-01"]
cq.groupby(["cell_id", "centroid_lat", "centroid_lon"]).agg(
    mean_maids=("n_maids", "mean"),
    max_maids=("n_maids", "max"),
    hari_aktif=("date", "nunique"),
).sort_values("mean_maids", ascending=False).head(10)

,,,mean_maids,max_maids,hari_aktif
cell_id,centroid_lat,centroid_lon,,,
E845_N18275,-7.800,110.299,"2,206.739",6459,69
E845_N18297,-7.700,110.300,"1,888.274",5554,73
E874_N18307,-7.655,110.431,"1,391.250",7899,24
E859_N18274,-7.805,110.363,"1,032.722",3481,90
E867_N18297,-7.701,110.399,956.967,3022,90
E865_N18299,-7.692,110.390,873.096,10971,83
E886_N18296,-7.705,110.485,872.600,993,5
E865_N18288,-7.741,110.390,717.200,1611,90
E867_N18275,-7.800,110.399,671.600,1797,90


In [ ]:
# Konsistensi dengan data per jam: apakah naik-turun jumlah sel aktif sejalan dengan total ping harian?
daily[["total_pings"]].join(daily_cells[["n_sel", "mean_maids"]]).corr()

,total_pings,n_sel,mean_maids
total_pings,1.000,0.836,0.900
n_sel,0.836,1.000,0.624
mean_maids,0.900,0.624,1.000


---
## C. Data asal–tujuan (OD): `od_home_visits` + `od_origin_homes`

### C.1 Struktur dan integritas

In [ ]:
display(od.head())
display(profil_kolom(od))
display(origin.head())
profil_kolom(origin)

,month,origin_kab_code,origin_kab_kota,origin_code,origin_kapanewon,dest_kab_code,dest_kab_kota,dest_code,dest_kapanewon,n_maids,reduced_quality
0,2021-12,34.01,KULON PROGO,34.01.01,TEMON,34.01,KULON PROGO,34.01.01,TEMON,569,False
1,2021-12,34.01,KULON PROGO,34.01.01,TEMON,34.01,KULON PROGO,34.01.02,WATES,85,False
2,2021-12,34.01,KULON PROGO,34.01.01,TEMON,34.01,KULON PROGO,34.01.03,PANJATAN,42,False
3,2021-12,34.01,KULON PROGO,34.01.01,TEMON,34.01,KULON PROGO,34.01.04,GALUR,16,False
4,2021-12,34.01,KULON PROGO,34.01.01,TEMON,34.01,KULON PROGO,34.01.05,LENDAH,8,False


,dtype,n_null,pct_null,n_unik,contoh
month,object,0,0.000,5,2021-12
origin_kab_code,object,0,0.000,5,34.01
origin_kab_kota,object,0,0.000,5,KULON PROGO
origin_code,object,0,0.000,78,34.01.01
origin_kapanewon,object,0,0.000,77,TEMON
dest_kab_code,object,0,0.000,5,34.01
dest_kab_kota,object,0,0.000,5,KULON PROGO
dest_code,object,0,0.000,78,34.01.01
dest_kapanewon,object,0,0.000,77,TEMON
n_maids,int64,0,0.000,566,569


,month,origin_kab_code,origin_kab_kota,origin_code,origin_kapanewon,n_homed_maids,n_homed_maids_daytime,reduced_quality
0,2021-12,34.01,KULON PROGO,34.01.01,TEMON,658,610,False
1,2021-12,34.01,KULON PROGO,34.01.02,WATES,1516,1364,False
2,2021-12,34.01,KULON PROGO,34.01.03,PANJATAN,169,149,False
3,2021-12,34.01,KULON PROGO,34.01.04,GALUR,328,311,False
4,2021-12,34.01,KULON PROGO,34.01.05,LENDAH,534,478,False


,dtype,n_null,pct_null,n_unik,contoh
month,object,0,0.000,5,2021-12
origin_kab_code,object,0,0.000,5,34.01
origin_kab_kota,object,0,0.000,5,KULON PROGO
origin_code,object,0,0.000,78,34.01.01
origin_kapanewon,object,0,0.000,77,TEMON
n_homed_maids,int64,0,0.000,307,658
n_homed_maids_daytime,int64,0,0.000,316,610
reduced_quality,bool,0,0.000,2,False


In [ ]:
od_key = ["month", "origin_code", "dest_code"]
print("Bulan                    :", sorted(od["month"].unique()))
print("Origin unik / dest unik  :", od["origin_code"].nunique(), "/", od["dest_code"].nunique())
print("Duplikat kunci OD        :", od.duplicated(od_key).sum())
print("Duplikat kunci origin    :", origin.duplicated(["month", "origin_code"]).sum())
print("n_maids minimum OD       :", od["n_maids"].min())
print("reduced_quality per bulan:", od.groupby("month")["reduced_quality"].unique().to_dict())
print("Null n_homed_maids_daytime:", origin["n_homed_maids_daytime"].isna().sum())
print("Daytime > homed (anomali):", (origin["n_homed_maids_daytime"] > origin["n_homed_maids"]).sum())

Bulan                    : ['2021-12', '2022-01', '2022-02', '2022-03', '2022-04']
Origin unik / dest unik  : 78 / 78
Duplikat kunci OD        : 0
Duplikat kunci origin    : 0
n_maids minimum OD       : 5
reduced_quality per bulan: {'2021-12': array([False]), '2022-01': array([False]), '2022-02': array([False]), '2022-03': array([ True]), '2022-04': array([ True])}
Null n_homed_maids_daytime: 0
Daytime > homed (anomali): 0


### C.2 Device ber-home per bulan (penyebut OD)

In [ ]:
orig_month = origin.groupby("month").agg(
    kapanewon=("origin_code", "nunique"),
    total_homed=("n_homed_maids", "sum"),
    total_homed_daytime=("n_homed_maids_daytime", "sum"),
    median_homed=("n_homed_maids", "median"),
    min_homed=("n_homed_maids", "min"),
)
orig_month["pct_teramati_siang"] = orig_month["total_homed_daytime"] / orig_month["total_homed"] * 100
orig_month["indeks_vs_des"] = orig_month["total_homed"] / orig_month["total_homed"].iloc[0] * 100
orig_month

,kapanewon,total_homed,total_homed_daytime,median_homed,min_homed,pct_teramati_siang,indeks_vs_des
month,,,,,,,
2021-12,78,122182,112475,809.000,26,92.055,100.000
2022-01,78,86013,77312,598.000,48,89.884,70.397
2022-02,78,77723,64269,435.500,30,82.690,63.612
2022-03,78,23185,12305,95.500,5,53.073,18.976
2022-04,78,14272,11624,63.500,10,81.446,11.681


In [ ]:
# Kapanewon dengan selisih homed vs daytime terbesar (device yang hanya terlihat malam)
origin.assign(pct_daytime=origin["n_homed_maids_daytime"] / origin["n_homed_maids"] * 100) \
      .sort_values("pct_daytime").head(10)[["month", "origin_kab_kota", "origin_kapanewon",
                                            "n_homed_maids", "n_homed_maids_daytime", "pct_daytime"]]

,month,origin_kab_kota,origin_kapanewon,n_homed_maids,n_homed_maids_daytime,pct_daytime
307,2022-03,YOGYAKARTA,GONDOMANAN,8016,1166,14.546
293,2022-03,SLEMAN,SLEMAN,2507,825,32.908
253,2022-03,BANTUL,BANTUL,526,185,35.171
298,2022-03,YOGYAKARTA,TEGALREJO,304,107,35.197
362,2022-04,SLEMAN,MINGGIR,105,54,51.429
285,2022-03,SLEMAN,SEYEGAN,433,226,52.194
284,2022-03,SLEMAN,MINGGIR,186,104,55.914
385,2022-04,YOGYAKARTA,GONDOMANAN,1976,1130,57.186
229,2022-02,YOGYAKARTA,GONDOMANAN,2785,1609,57.774
376,2022-04,YOGYAKARTA,TEGALREJO,114,70,61.404


In [ ]:
# Validasi penyebut terhadap populasi resmi (tanpa Gondomanan)
o = origin.merge(homepop[["KAB_KOTA", "KEC_KAPN", "pop_actual"]],
                 left_on=["origin_kab_kota", "origin_kapanewon"], right_on=["KAB_KOTA", "KEC_KAPN"], how="left")
print("Baris origin tanpa data populasi:", o["pop_actual"].isna().sum())
o2 = o[o["origin_kapanewon"] != "GONDOMANAN"]
pd.DataFrame({
    m: {
        "n": len(g),
        "pearson_linear": g["n_homed_maids"].corr(g["pop_actual"]),
        "pearson_log": np.log(g["n_homed_maids"]).corr(np.log(g["pop_actual"])),
        "spearman": g["n_homed_maids"].corr(g["pop_actual"], method="spearman"),
    }
    for m, g in o2.groupby("month")
}).T

Baris origin tanpa data populasi: 0


,n,pearson_linear,pearson_log,spearman
2021-12,77.000,0.822,0.738,0.730
2022-01,77.000,0.868,0.739,0.732
2022-02,77.000,0.835,0.684,0.662
2022-03,77.000,0.510,0.571,0.518
2022-04,77.000,0.689,0.570,0.518


### C.3 Ukuran dan komposisi matriks OD

In [ ]:
n_zone = 78
od["intra"] = od["origin_code"] == od["dest_code"]
od["jenis"] = np.select(
    [od["intra"], od["origin_kab_code"] == od["dest_kab_code"]],
    ["intra-kapanewon", "antar-kapanewon (1 kab)"], default="antar-kabupaten")

od_month = od.groupby("month").agg(
    pasangan=("n_maids", "size"),
    origin_unik=("origin_code", "nunique"),
    dest_unik=("dest_code", "nunique"),
    median_maids=("n_maids", "median"),
    max_maids=("n_maids", "max"),
)
od_month["pct_dari_78x78"] = od_month["pasangan"] / (n_zone * n_zone) * 100
od_month

,pasangan,origin_unik,dest_unik,median_maids,max_maids,pct_dari_78x78
month,,,,,,
2021-12,2595,78,78,18.000,7961,42.653
2022-01,2624,78,78,16.000,6644,43.130
2022-02,2653,78,78,16.000,5532,43.606
2022-03,1202,78,78,11.000,779,19.757
2022-04,820,78,78,10.000,913,13.478


In [ ]:
# Komposisi jumlah pasangan dan jumlah device per jenis aliran
komp = od.groupby(["month", "jenis"])["n_maids"].agg(pasangan="size", total_maids="sum").unstack()
komp

pasangan                                             total_maids                                        
jenis   antar-kabupaten antar-kapanewon (1 kab) intra-kapanewon antar-kabupaten antar-kapanewon (1 kab) intra-kapanewon
month                                                                                                                  
2021-12            1728                     789              78           68014                   75957          105170
2022-01            1738                     808              78           56162                   54004           71745
2022-02            1766                     809              78           56207                   44752           57846
2022-03             628                     496              78           10218                    8410           10942
2022-04             416                     326              78            6835                    5349           10795

In [ ]:
# Proporsi device yang hanya terlihat di wilayah sendiri vs bepergian
od = od.merge(origin[["month", "origin_code", "n_homed_maids", "n_homed_maids_daytime"]],
              on=["month", "origin_code"], how="left")
print("Baris OD tanpa penyebut:", od["n_homed_maids_daytime"].isna().sum())
od["share_visit"] = od["n_maids"] / od["n_homed_maids_daytime"]
print("Baris share_visit > 1 (anomali):", (od["share_visit"] > 1).sum())

print("\nShare device yang terlihat di kapanewon rumahnya sendiri pada siang hari:")
od[od["intra"]].groupby("month")["share_visit"].describe()

Baris OD tanpa penyebut: 0
Baris share_visit > 1 (anomali): 0

Share device yang terlihat di kapanewon rumahnya sendiri pada siang hari:


,count,mean,std,min,25%,50%,75%,max
month,,,,,,,,
2021-12,78.000,0.926,0.052,0.752,0.913,0.943,0.958,0.982
2022-01,78.000,0.927,0.045,0.771,0.922,0.939,0.951,0.989
2022-02,78.000,0.920,0.078,0.455,0.913,0.940,0.962,1.000
2022-03,78.000,0.951,0.082,0.504,0.955,0.977,1.000,1.000
2022-04,78.000,0.948,0.053,0.767,0.934,0.959,0.984,1.000


In [ ]:
print("Statistik deskriptif n_maids per jenis aliran (Des-Feb):")
od[~od["reduced_quality"]].groupby("jenis")["n_maids"].describe(percentiles=[.25, .5, .75, .95])

Statistik deskriptif n_maids per jenis aliran (Des-Feb):


,count,mean,std,min,25%,50%,75%,95%,max
jenis,,,,,,,,,
antar-kabupaten,"5,232.000",34.477,77.020,5.000,7.000,13.000,29.000,132.450,"1,394.000"
antar-kapanewon (1 kab),"2,406.000",72.616,152.934,5.000,12.000,28.000,65.000,284.750,"2,661.000"
intra-kapanewon,234.000,"1,003.252","1,398.360",16.000,180.000,501.000,"1,122.500","3,965.300","7,961.000"


### C.4 Tujuan dan pasangan paling kuat

Karena tiap device hanya punya **satu** home per bulan, `n_maids` boleh dijumlahkan **antar origin** untuk satu tujuan
(hasilnya = device dari luar yang mengunjungi tujuan itu, minus pasangan yang ter-suppress).
Untuk dibandingkan antar bulan, hasilnya dinormalisasi dengan total device ber-home di bulan tersebut.

In [ ]:
inter = od[~od["intra"]]
inter_hq = inter[~inter["reduced_quality"]]

tujuan = inter.groupby(["dest_kab_kota", "dest_kapanewon", "month"])["n_maids"].sum().unstack()
tujuan_pct = tujuan.div(origin.groupby("month")["n_homed_maids_daytime"].sum()[tujuan.columns], axis=1) * 100
tujuan_pct["mean_des_feb"] = tujuan_pct[["2021-12", "2022-01", "2022-02"]].mean(axis=1)
print("10 tujuan paling banyak dikunjungi device dari luar (% dari seluruh device ber-home)")
tujuan_pct.sort_values("mean_des_feb", ascending=False).head(10)

10 tujuan paling banyak dikunjungi device dari luar (% dari seluruh device ber-home)


month                         2021-12  2022-01  2022-02  2022-03  2022-04  mean_des_feb
dest_kab_kota dest_kapanewon                                                           
SLEMAN        DEPOK             9.449   11.820   12.079   10.874    7.992        11.116
BANTUL        BANGUNTAPAN       6.961    7.356    7.453    7.095    3.725         7.257
SLEMAN        NGAGLIK           7.307    6.545    7.056    5.916    5.102         6.969
              GAMPING           7.261    5.973    5.682    5.380    3.501         6.306
YOGYAKARTA    GONDOKUSUMAN      4.467    6.211    7.741    5.908    5.093         6.140
SLEMAN        MLATI             4.759    6.242    7.045    6.306    5.093         6.016
YOGYAKARTA    UMBULHARJO        5.086    6.392    6.448    5.486    4.697         5.976
BANTUL        KASIHAN           5.351    5.479    5.732    5.104    3.286         5.521
SLEMAN        PAKEM             6.562    3.438    4.237    3.308    1.557         4.746
              NGEMPLAK          4.399    5.125    4.204    2.983    2.228         4.576

In [ ]:
# Jumlah kapanewon asal yang mengirim pengunjung (>=5 device) ke tiap tujuan, Des-Feb
jangkauan = inter_hq.groupby(["dest_kab_kota", "dest_kapanewon", "month"])["origin_code"].nunique().unstack()
jangkauan["mean"] = jangkauan.mean(axis=1)
jangkauan.sort_values("mean", ascending=False).head(10)

month                         2021-12  2022-01  2022-02   mean
dest_kab_kota dest_kapanewon                                  
SLEMAN        DEPOK                70       73       69 70.667
BANTUL        BANGUNTAPAN          67       67       66 66.667
              KASIHAN              70       65       60 65.000
SLEMAN        NGAGLIK              68       58       59 61.667
              GAMPING              66       60       56 60.667
              SLEMAN               56       62       61 59.667
YOGYAKARTA    GONDOMANAN           58       56       64 59.333
SLEMAN        MLATI                58       57       58 57.667
YOGYAKARTA    UMBULHARJO           59       56       58 57.667
              GONDOKUSUMAN         55       60       56 57.000

In [ ]:
# Pasangan antar-kapanewon dengan proporsi kunjungan tertinggi (rata-rata Des-Feb)
inter_hq.groupby(["origin_kab_kota", "origin_kapanewon", "dest_kab_kota", "dest_kapanewon"]).agg(
    mean_share_visit=("share_visit", "mean"),
    mean_maids=("n_maids", "mean"),
    bulan_muncul=("month", "nunique"),
).sort_values("mean_share_visit", ascending=False).head(15)

mean_share_visit  mean_maids  bulan_muncul
origin_kab_kota origin_kapanewon dest_kab_kota dest_kapanewon                                            
YOGYAKARTA      KOTAGEDE         BANTUL        BANGUNTAPAN                0.660     638.667             3
                                 YOGYAKARTA    UMBULHARJO                 0.617     596.333             3
KULON PROGO     GALUR            KULON PROGO   LENDAH                     0.505     121.333             3
YOGYAKARTA      PAKUALAMAN       YOGYAKARTA    DANUREJAN                  0.495     174.000             3
                                               MERGANGSAN                 0.451     159.333             3
                GONDOMANAN       SLEMAN        SLEMAN                     0.433     497.333             3
GUNUNGKIDUL     TANJUNGSARI      GUNUNGKIDUL   WONOSARI                   0.429      29.333             3
KULON PROGO     PENGASIH         KULON PROGO   WATES                      0.414     338.667             3
YOGYAKARTA      WIROBRAJAN       BANTUL        KASIHAN                    0.379      47.333             3
                MANTRIJERON      BANTUL        KASIHAN                    0.366      55.000             3
SLEMAN          BERBAH           BANTUL        BANGUNTAPAN                0.365     365.667             3
GUNUNGKIDUL     NGLIPAR          GUNUNGKIDUL   PLAYEN                     0.348      69.333             3
YOGYAKARTA      MERGANGSAN       YOGYAKARTA    UMBULHARJO                 0.347     133.000             3
                MANTRIJERON      BANTUL        SEWON                      0.341      53.333             3
                PAKUALAMAN       YOGYAKARTA    UMBULHARJO                 0.336     115.333             3

In [ ]:
# Resiprositas: apakah aliran i->j diikuti aliran j->i?
p = inter[["month", "origin_code", "dest_code", "n_maids"]]
rec = p.merge(p, left_on=["month", "origin_code", "dest_code"],
              right_on=["month", "dest_code", "origin_code"], suffixes=("_ij", "_ji"))
pd.DataFrame({
    m: {
        "pasangan_antar": (p["month"] == m).sum(),
        "pct_ada_arah_balik": len(g) / (p["month"] == m).sum() * 100,
        "pearson_log_ij_ji": np.corrcoef(np.log(g["n_maids_ij"]), np.log(g["n_maids_ji"]))[0, 1],
    }
    for m, g in rec.groupby("month")
}).T

,pasangan_antar,pct_ada_arah_balik,pearson_log_ij_ji
2021-12,"2,517.000",80.652,0.806
2022-01,"2,546.000",82.404,0.808
2022-02,"2,575.000",80.544,0.777
2022-03,"1,124.000",73.132,0.673
2022-04,742.000,71.429,0.721


---
## D. `home_maids_vs_population_kapanewon` — device ber-home vs populasi

In [ ]:
display(homepop.head())
display(profil_kolom(homepop))
print("Duplikat (kab, kapanewon):", homepop.duplicated(["KAB_KOTA", "KEC_KAPN"]).sum())
print("Ratio = n_users / pop_actual konsisten:",
      np.allclose(homepop["ratio"], homepop["n_users"] / homepop["pop_actual"]))

# Cek kecocokan nama dengan data lain
nama_hp = set(zip(homepop["KAB_KOTA"], homepop["KEC_KAPN"]))
nama_hr = set(zip(hourly["kab_kota"], hourly["kapanewon"]))
print("Ada di homepop tapi tidak di hourly:", nama_hp - nama_hr)
print("Ada di hourly tapi tidak di homepop:", nama_hr - nama_hp)

,KAB_KOTA,KEC_KAPN,n_users,pop_actual,ratio
0,YOGYAKARTA,GONDOMANAN,28874,"12,850.000",2.247
1,SLEMAN,PAKEM,20241,"37,580.000",0.539
2,SLEMAN,DEPOK,39609,"130,980.000",0.302
3,SLEMAN,SEYEGAN,15469,"51,860.000",0.298
4,YOGYAKARTA,JETIS,6922,"23,480.000",0.295


,dtype,n_null,pct_null,n_unik,contoh
KAB_KOTA,object,0,0.000,5,YOGYAKARTA
KEC_KAPN,object,0,0.000,77,GONDOMANAN
n_users,int64,0,0.000,78,28874
pop_actual,float64,0,0.000,78,"12,850.000"
ratio,float64,0,0.000,78,2.247


Duplikat (kab, kapanewon): 0
Ratio = n_users / pop_actual konsisten: True
Ada di homepop tapi tidak di hourly: set()
Ada di hourly tapi tidak di homepop: set()


In [ ]:
d = homepop[["n_users", "pop_actual", "ratio"]].describe(percentiles=[.1, .25, .5, .75, .9]).T
d["skew"] = homepop[["n_users", "pop_actual", "ratio"]].skew()
d

,count,mean,std,min,10%,25%,50%,75%,90%,max,skew
n_users,78.000,"6,556.654","8,168.749",256.000,672.500,"1,299.500","3,337.000","7,364.750","16,756.300","39,609.000",2.098
pop_actual,78.000,"47,279.231","26,514.827","9,190.000","23,949.000","30,540.000","37,975.000","55,547.500","87,279.000","130,980.000",1.408
ratio,78.000,0.139,0.260,0.009,0.020,0.041,0.090,0.147,0.271,2.247,7.179


In [ ]:
# Outlier rasio dengan metode IQR
q1, q3 = homepop["ratio"].quantile([.25, .75])
iqr = q3 - q1
batas_bawah, batas_atas = q1 - 1.5 * iqr, q3 + 1.5 * iqr
print(f"Q1={q1:.3f}  Q3={q3:.3f}  IQR={iqr:.3f}  batas atas={batas_atas:.3f}")
homepop[(homepop["ratio"] > batas_atas) | (homepop["ratio"] < batas_bawah)].sort_values("ratio", ascending=False)

Q1=0.041  Q3=0.147  IQR=0.106  batas atas=0.307


,KAB_KOTA,KEC_KAPN,n_users,pop_actual,ratio
0,YOGYAKARTA,GONDOMANAN,28874,"12,850.000",2.247
1,SLEMAN,PAKEM,20241,"37,580.000",0.539


In [ ]:
def korelasi(df, x, y):
    return pd.Series({
        "n": len(df),
        "pearson_linear": df[x].corr(df[y]),
        "pearson_log": np.log(df[x]).corr(np.log(df[y])),
        "spearman": df[x].corr(df[y], method="spearman"),
    })

pd.DataFrame({
    "semua": korelasi(homepop, "n_users", "pop_actual"),
    "tanpa GONDOMANAN": korelasi(homepop[homepop["KEC_KAPN"] != "GONDOMANAN"], "n_users", "pop_actual"),
}).T

,n,pearson_linear,pearson_log,spearman
semua,78.000,0.723,0.600,0.614
tanpa GONDOMANAN,77.000,0.820,0.688,0.672


In [ ]:
# Per kabupaten/kota (n_users boleh dijumlah: tiap device punya satu home)
per_kab = homepop.groupby("KAB_KOTA").agg(
    kapanewon=("KEC_KAPN", "size"),
    n_users=("n_users", "sum"),
    pop_actual=("pop_actual", "sum"),
    ratio_median=("ratio", "median"),
    ratio_min=("ratio", "min"),
    ratio_max=("ratio", "max"),
)
per_kab["ratio_agregat"] = per_kab["n_users"] / per_kab["pop_actual"]
per_kab.sort_values("ratio_agregat", ascending=False)

,kapanewon,n_users,pop_actual,ratio_median,ratio_min,ratio_max,ratio_agregat
KAB_KOTA,,,,,,,
YOGYAKARTA,14,87734,"375,280.000",0.145,0.033,2.247,0.234
SLEMAN,17,238777,"1,134,150.000",0.163,0.075,0.539,0.211
BANTUL,17,124529,"991,890.000",0.103,0.019,0.220,0.126
KULON PROGO,12,32578,"438,190.000",0.075,0.027,0.148,0.074
GUNUNGKIDUL,18,27801,"748,270.000",0.022,0.009,0.113,0.037


---
## E. `validation_night_presence_kapanewon` — presence malam hari

In [ ]:
display(profil_kolom(night))
print(night["period"].value_counts())
night.groupby("period")[["night_presence", "slots_published_pct"]].describe().T

,dtype,n_null,pct_null,n_unik,contoh
period,object,0,0.000,2,Dec 2021 - Apr 2022
kab_kota,object,0,0.000,5,BANTUL
kapanewon,object,0,0.000,77,BAMBANGLIPURO
night_presence,float64,0,0.000,156,23.980
slots_published_pct,float64,0,0.000,111,60.817
n_users,int64,0,0.000,78,4497
pop_actual,float64,0,0.000,78,"41,070.000"


period
Dec 2021 - Apr 2022    78
Dec 2021 - Jan 2022    78
Name: count, dtype: int64


period                     Dec 2021 - Apr 2022  Dec 2021 - Jan 2022
night_presence      count               78.000               78.000
                    mean                34.462               41.665
                    std                 37.884               50.806
                    min                  5.814                5.632
                    25%                 10.317               10.322
                    50%                 21.577               21.339
                    75%                 37.799               46.848
                    max                200.513              278.016
slots_published_pct count               78.000               78.000
                    mean                65.843               82.923
                    std                 27.221               28.118
                    min                  3.201                5.108
                    25%                 51.490               77.419
                    50%                 69.316               98.925
                    75%                 90.315              100.000
                    max                100.000              100.000

In [ ]:
# Perbandingan dua periode per kapanewon
nv = night.pivot_table(index=["kab_kota", "kapanewon"], columns="period",
                       values=["night_presence", "slots_published_pct"])
nv.columns = [f"{a}|{b}" for a, b in nv.columns]
print("Kapanewon dengan slot malam terpublikasi < 20% (periode penuh):")
nv[nv["slots_published_pct|Dec 2021 - Apr 2022"] < 20].sort_values("slots_published_pct|Dec 2021 - Apr 2022")

Kapanewon dengan slot malam terpublikasi < 20% (periode penuh):


night_presence|Dec 2021 - Apr 2022  night_presence|Dec 2021 - Jan 2022  slots_published_pct|Dec 2021 - Apr 2022  slots_published_pct|Dec 2021 - Jan 2022
kab_kota    kapanewon                                                                                                                                                            
GUNUNGKIDUL GIRISUBO                                  5.828                               5.632                                    3.201                                    5.108
            RONGKOP                                   5.814                               5.897                                    4.746                                    7.796
            SAPTOSARI                                 6.150                               6.220                                    6.623                                   11.022
            TEPUS                                     6.108                               6.192                                    7.174                                   13.978
            TANJUNGSARI                               6.153                               6.021                                    9.382                                   12.634
            PURWOSARI                                 6.283                               6.081                                   10.927                                   16.667
            PANGGANG                                  6.988                               6.699                                   17.770                                   27.688
            NGAWEN                                    6.862                               6.810                                   18.433                                   26.882

In [ ]:
# Korelasi presence malam vs populasi dan vs device ber-home, per periode, tanpa Gondomanan
n2 = night[night["kapanewon"] != "GONDOMANAN"]
pd.DataFrame({
    (per, y): korelasi(g, "night_presence", y)
    for per, g in n2.groupby("period")
    for y in ["pop_actual", "n_users"]
}).T

n  pearson_linear  pearson_log  spearman
Dec 2021 - Apr 2022 pop_actual 77.000           0.825        0.702     0.661
                    n_users    77.000           0.988        0.977     0.986
Dec 2021 - Jan 2022 pop_actual 77.000           0.824        0.718     0.679
                    n_users    77.000           0.987        0.975     0.983

---
## F. File ringkasan — cek konsistensi dengan data

In [ ]:
display(tier1)
display(od_sum)

cek = pd.DataFrame({
    "nilai_di_ringkasan": [
        tier1.loc[tier1["product"] == "A_cell500m_daily", "rows_published"].iloc[0],
        tier1.loc[tier1["product"] == "B_kapanewon_hourly", "rows_published"].iloc[0],
        tier1.loc[tier1["product"] == "A_cell500m_daily", "n_cells_published"].iloc[0],
        od_sum["od_rows_published"].iloc[0],
        od_sum["origin_rows_published"].iloc[0],
    ],
    "nilai_dihitung": [len(cell), len(hourly), cell["cell_id"].nunique(), len(od), len(origin)],
}, index=["baris cell500m", "baris kapanewon_hourly", "jumlah sel", "baris OD", "baris origin"])
cek["cocok"] = cek["nilai_di_ringkasan"] == cek["nilai_dihitung"]
cek

,pings_inside_diy_pct,product,k,rows_total,rows_published,rows_suppressed_pct,maid_presence_suppressed_pct,n_cells_published
0,100.000,A_cell500m_daily,5,518176,183548,64.600,6.600,5715
1,100.000,B_kapanewon_hourly,5,273979,230418,15.900,0.700,78


,k,min_nights,od_rows_total,od_rows_published,od_rows_suppressed_pct,inter_zonal_flow_kept_pct,origin_rows_published
0,5,2,19263,9894,48.600,95.300,390


,nilai_di_ringkasan,nilai_dihitung,cocok
baris cell500m,183548,183548,True
baris kapanewon_hourly,230418,230418,True
jumlah sel,5715,5715,True
baris OD,9894,9894,True
baris origin,390,390,True


---
## G. Ringkasan temuan EDA


**Kualitas dan struktur data**
1. Semua file konsisten dengan README: (cek bagian F) jumlah baris, 78 kapanewon, 151 hari, 5.715 sel, tidak ada duplikat kunci.
2. Nama kapanewon tidak unik (JETIS di Bantul dan Kota Yogyakarta), jadi join harus memakai kode atau pasangan kabupaten + kapanewon.
3. Suppression membuat **18,5% slot kapanewon × jam** dan **~65% baris sel × hari** hilang. Kehilangan terpusat di **Gunungkidul dan Kulon Progo**, pada **jam malam**, dan pada **Maret–April** (bagian A.2).

**Perubahan sampel dari waktu ke waktu**
4. Volume data turun tajam: rata-rata ping per hari di April hanya **10,2%** dari Desember (A.4). Device ber-home OD turun dari **122.182** (Des) ke **14.272** (Apr) (C.2).
5. Titik patah terlihat pada **1 Januari** dan **16 Februari 2022**, ditambah anomali satu hari pada **8 Maret 2022** (A.4).
6. Konsekuensi: perbandingan antar waktu **wajib dinormalisasi**, dan analisis kuantitatif sebaiknya memakai **Des 2021 – Feb 2022**.

**Sebaran nilai**
7. `n_maids` sangat menceng ke kanan (skew **4,782**). Median jauh di bawah mean, didominasi segelintir kapanewon/sel perkotaan. Untuk analisis lanjutan pertimbangkan transformasi log.
8. Kehadiran device sangat terkonsentrasi: **79,9%** device-hari berada di 10% sel teratas (B.3).

**Pola awal**
9. Diurnal: titik terendah sekitar jam **3**, naik mulai jam 05, datar sekitar jam 11–20 (A.5).
10. Mingguan: hari **minggu** paling rendah setelah normalisasi (A.5).
11. Spasial: presence tertinggi di **Depok** (A.6). Rasio siang/malam tertinggi di **Playen** (pusat kegiatan), terendah di **Gondomanan** (permukiman).
12. OD: **39%** (menggunakan data Des–Feb: sekitar 36–42%. Maret–April tidak dipakai karena kualitas datanya rendah) aliran berupa intra-kapanewon. Tujuan antar-kapanewon terpopuler adalah **Depok** (C.4). Aliran cukup resiprokal (r log ≈ **0,78–0,81 (Des–Feb)**).

**Representativitas**
13. Device ber-home berkorelasi dengan populasi (r log ≈ **0,688** tanpa Gondomanan) (D).
14. **Gondomanan** adalah outlier kuat (rasio device/penduduk ≈ 2,25), kemungkinan karena wisatawan/pengunjung. Perlu dikeluarkan atau dibahas terpisah.

**Implikasi untuk tahap berikutnya**
- Normalisasi (share per jam/hari) sebelum membandingkan waktu.
- Baris hilang ≠ 0; jangan diisi nol tanpa alasan.
- Jangan menjumlahkan `n_maids` lintas jam/wilayah/hari (kecuali OD lintas origin untuk satu tujuan).
- Prioritaskan Des 2021 – Feb 2022, dan tandai kapanewon dengan kelengkapan rendah.